In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [3]:
# Load dataset
df = pd.read_csv('student_performance_5000.csv')



In [4]:
# Create single target variable for grade
def grade_label(row):
    if row['Fail'] == 1:
        return 0
    elif row['Grade3'] == 1:
        return 1
    elif row['Grade2'] == 1:
        return 2
    else:
        return 3

df['Grade'] = df.apply(grade_label, axis=1)

In [5]:
# Drop original grade columns
df.drop(['Grade3','Grade2','Grade1','Fail'], axis=1, inplace=True)

In [6]:
# Features and target
X = df.drop('Grade', axis=1)
y = df['Grade']

In [7]:
# Categorical columns
cat_cols = ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']
num_cols = ['math score', 'reading score', 'writing score']


In [8]:

# Preprocessing pipeline
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(), cat_cols)
], remainder='passthrough')


In [9]:
# Model pipeline
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])



In [10]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:

# Train
clf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat', OneHotEncoder(),
                                                  ['gender', 'race/ethnicity',
                                                   'parental level of '
                                                   'education',
                                                   'lunch',
                                                   'test preparation '
                                                   'course'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [12]:

# Predict
y_pred = clf.predict(X_test)

In [13]:
# Report
print(classification_report(y_test, y_pred, target_names=['Fail','Grade3','Grade2','Grade1']))

              precision    recall  f1-score   support

        Fail       1.00      0.18      0.31        11
      Grade3       0.89      0.83      0.86       307
      Grade2       0.85      0.96      0.90       611
      Grade1       1.00      0.31      0.47        71

    accuracy                           0.87      1000
   macro avg       0.94      0.57      0.64      1000
weighted avg       0.88      0.87      0.85      1000



In [14]:
import joblib
from sklearn.metrics import accuracy_score

In [15]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy:.4f}")


Model accuracy: 0.8670


In [16]:
# Save the model to a file
joblib.dump(clf, 'student_grade_predictor.pkl')
print("Model saved as 'student_grade_predictor.pkl'")

Model saved as 'student_grade_predictor.pkl'
